In [1]:
import pandas as pd
import numpy as np
import warnings
import re
import matplotlib.pyplot as plt
import scienceplots
plt.style.use(['science', 'notebook', 'grid'])
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet
import ast

import copy
from collections import Counter
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv('sonas.csv', sep=',', encoding='UTF-8')
df.insert(loc=4, column='full_text_clean', value=df['Full Text'])

df['full_text_clean'] = df['full_text_clean'].str.replace(r'\n', ' ', regex=True)  # Remove newlines
df['full_text_clean'] = df['full_text_clean'].str.lower()  # Convert to lowercase
df['full_text_clean'] = df['full_text_clean'].str.replace(r'[^\w\s]+', ' ', regex=True)  # Remove punctuation
df['full_text_clean'] = df['full_text_clean'].str.replace(r'[0-9]+', '', regex=True)  # Remove numbers
df['full_text_clean'] = df['full_text_clean'].str.replace(r' +', ' ', regex=True).str.strip()  # Remove extra spaces
df['full_text_clean'] = df['full_text_clean'].apply(word_tokenize) # Apply tokenization

stop_words = set(stopwords.words('english'))
df['full_text_clean'] = df['full_text_clean'].apply(lambda x: [word for word in x if word not in stop_words])

df

,Title,Month,Year,Full Text,full_text_clean
0,President Kassym-Jomart Tokayev’s State of the...,September,2024,Dear Members of Parliament!\n\nCongratulations...,"[dear, members, parliament, congratulations, o..."
1,President Kassym-Jomart Tokayev`s State of the...,September,2023,"Dear Compatriots, Distinguished Members of Par...","[dear, compatriots, distinguished, members, pa..."
2,President Kassym-Jomart Tokayev’s State of the...,September,2022,PRESIDENT KASSYM-JOMART TOKAYEV’S\n\nSTATE OF ...,"[president, kassym, jomart, tokayev, state, na..."
3,State-of-the-Nation Address by President of th...,March,2022,NEW KAZAKHSTAN: THE PATH OF RENEWAL AND MODERN...,"[new, kazakhstan, path, renewal, modernisation..."
4,State of the Nation Address by President of th...,September,2021,UNITY OF THE PEOPLE AND SYSTEMIC REFORMS ARE A...,"[unity, people, systemic, reforms, solid, foun..."
5,President of Kazakhstan Kassym-Jomart Tokayev’...,September,2020,KAZAKHSTAN IN A NEW REALITY: TIME FOR ACTION\n...,"[kazakhstan, new, reality, time, action, work,..."
6,President of Kazakhstan Kassym-Jomart Tokayev’...,September,2019,President Kassym-Jomart Tokayev’s State of the...,"[president, kassym, jomart, tokayev, state, na..."
7,State of the Nation Address of President of th...,October,2018,GROWING WELFARE OF KAZAKH CITIZENS: INCREASE I...,"[growing, welfare, kazakh, citizens, increase,..."
8,State of the Nation Address by the President o...,January,2018,NEW OPPORTUNITIES UNDER THE FOURTH INDUSTRIAL ...,"[new, opportunities, fourth, industrial, revol..."
9,The President of Kazakhstan Nursultan Nazarbay...,January,2017,"""Third Modernization of Kazakhstan: Global Com...","[third, modernization, kazakhstan, global, com..."


In [3]:
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN 

l = WordNetLemmatizer()
df['full_text_clean'] = df['full_text_clean'].apply(
    lambda tokens: [
        l.lemmatize(word, get_wordnet_pos(pos_tag([word])[0][1]))
        for word in tokens
    ]
)
#df['full_text_clean'] = df['tokens'].apply(lambda x: ' '.join(x))

df

,Title,Month,Year,Full Text,full_text_clean
0,President Kassym-Jomart Tokayev’s State of the...,September,2024,Dear Members of Parliament!\n\nCongratulations...,"[dear, member, parliament, congratulation, ope..."
1,President Kassym-Jomart Tokayev`s State of the...,September,2023,"Dear Compatriots, Distinguished Members of Par...","[dear, compatriot, distinguish, member, parlia..."
2,President Kassym-Jomart Tokayev’s State of the...,September,2022,PRESIDENT KASSYM-JOMART TOKAYEV’S\n\nSTATE OF ...,"[president, kassym, jomart, tokayev, state, na..."
3,State-of-the-Nation Address by President of th...,March,2022,NEW KAZAKHSTAN: THE PATH OF RENEWAL AND MODERN...,"[new, kazakhstan, path, renewal, modernisation..."
4,State of the Nation Address by President of th...,September,2021,UNITY OF THE PEOPLE AND SYSTEMIC REFORMS ARE A...,"[unity, people, systemic, reform, solid, found..."
5,President of Kazakhstan Kassym-Jomart Tokayev’...,September,2020,KAZAKHSTAN IN A NEW REALITY: TIME FOR ACTION\n...,"[kazakhstan, new, reality, time, action, work,..."
6,President of Kazakhstan Kassym-Jomart Tokayev’...,September,2019,President Kassym-Jomart Tokayev’s State of the...,"[president, kassym, jomart, tokayev, state, na..."
7,State of the Nation Address of President of th...,October,2018,GROWING WELFARE OF KAZAKH CITIZENS: INCREASE I...,"[grow, welfare, kazakh, citizen, increase, inc..."
8,State of the Nation Address by the President o...,January,2018,NEW OPPORTUNITIES UNDER THE FOURTH INDUSTRIAL ...,"[new, opportunity, fourth, industrial, revolut..."
9,The President of Kazakhstan Nursultan Nazarbay...,January,2017,"""Third Modernization of Kazakhstan: Global Com...","[third, modernization, kazakhstan, global, com..."


In [4]:
import copy
df_test = copy.deepcopy(df)
df_test.head()

,Title,Month,Year,Full Text,full_text_clean
0,President Kassym-Jomart Tokayev’s State of the...,September,2024,Dear Members of Parliament!\n\nCongratulations...,"[dear, member, parliament, congratulation, ope..."
1,President Kassym-Jomart Tokayev`s State of the...,September,2023,"Dear Compatriots, Distinguished Members of Par...","[dear, compatriot, distinguish, member, parlia..."
2,President Kassym-Jomart Tokayev’s State of the...,September,2022,PRESIDENT KASSYM-JOMART TOKAYEV’S\n\nSTATE OF ...,"[president, kassym, jomart, tokayev, state, na..."
3,State-of-the-Nation Address by President of th...,March,2022,NEW KAZAKHSTAN: THE PATH OF RENEWAL AND MODERN...,"[new, kazakhstan, path, renewal, modernisation..."
4,State of the Nation Address by President of th...,September,2021,UNITY OF THE PEOPLE AND SYSTEMIC REFORMS ARE A...,"[unity, people, systemic, reform, solid, found..."


In [5]:
total_word_count_before = df_test['Full Text'].str.replace('\n', ' ', regex=False).str.split().explode().count()

print("Total word count:", total_word_count_before)

Total word count: 218669


In [6]:
custom_stop = ["law", "new", "pay", "way", "key", "let", "u", "use", "aim", "one", "set", "may", "yet", "ten", "two", "non", "raw", 
"km", "per", "end", "npp", "air", "ust", "top", "e", "lay", "sea", "ton", "bad", "go", 
"put", "mp", "pre", "led", "say", "act", "ass", "age", "die", "day", "men", "son", 
"lie", "sow", "un", "see", "lot", "era", "met", "gdp", "hit", "wto", "low", "arm", 
"esg", "le", "dry", "xi", "due", "fit", "sum", "ipo", "net", "ask", "vat", "ups", 
"red", "de", "tap", "man", "pas", "th", "fee", "car", "fix", "get", "rid", "ppp", 
"sez", "co", "map", "el", "eye", "hr", "yes", "mid", "run", "far", "war", "fun", 
"ban", "apk", "ago", "ngo", "fur", "saw", "bet", "і", "cut", "usd", "ii", "ill", 
"iii", "tve", "iv", "mw", "v", "vi", "vii", "six", "art", "ci", "sit", "nur", "win", 
"bid", "sme", "ivf", "gap", "big", "bed", "lag", "apo", "al", "got", "ho", "jet", 
"eco", "g", "dog", "hot", "mci", "etc", "c", "bar", "phc", "usa", "st", "eu", 
"tie", "yel", "boy", "re", "kwh", "күй", "bio", "b", "kzt", "nd", "rd", "jac", "sco", "jsc", 
"sub", "eve", "nce", "n", "fdi", "via", "box", "ore", "r", "arc", "oic", "bit", "god", 
"bug", "buy", "ce", "ski", "phd", "hiv", "row", "cec", "xxi", "pit", "tpp", "hpp", 
"pdp", "jvc", "pin", "joy", "aul", "h", "mr", "cum", "sec", "ad", "hoc", "us","pro",
"con", "spk", "sbc", "hei", "mix", "tv", "add", "try", "iso", "id", "opt", "geo", 
"bn", "mln", "pc", "luc", "xx", "kg", "tc", "xvi", "jew", "sag", "dy", "rob", "quo", 
"fed", "sun", "eat", "prc", "uno", "imf", "lax", "bi", "jts", "leg", "dug", "min", "fax", "w", "tg", "cpc", "bin", "bud"]

other_stop = [
    "kazakhstan", "ofkazakhstan", "year", "must", "need", "also", "first", "become", "would", "term",
    "right", "therefore", "high", "take", "many", "make", "kazakh", "however", "based",
    "second", "strengthening", "astana", "full", "kazakhstani", "achieve", "especially",
    "thousand", "kassym", "jomart", "tokayev", "nazarbayev", "nursultan", "establish", "even",
    "creating", "professional", "training", "give", "mean", "like stage", "percent",
    "significant", "among", "whole", "come", "taking", "ensuring", "fully", "example", "able",
    "general", "additional", "always", "following", "reduce", "improving", "working", "fourth",
    "five", "another", "cent", "still", "done", "move", "january", "decade", "code", "secondly",
    "sized", "living", "least", "thus", "appropriate", "paid", "remains", "amount", "almaty", "thing",
    "firstly", "question", "existing", "total", "republic", "around", "every", "long", "within",
    "already", "basis", "without", "three","good", "body", "address", "jumystyń", "kózin", "baılyqtyń", "ózin", 
    "besіgindi", "besіgі", "ғаламтор", "қолтырауын", "сандық", "otherы", "besіgі",
    "system", "well", "next", "part", "together", "third", "last", "made", 
    "towards", "serious", "since", "could", "much", "almost", "ever", "never", "along", "everything",
    "shall", "trillion", "thirdly", "particular", "regard", "provided", "meet", "given", "taken", "dear", "member",
    "compatriot", "congratulation", "parliament", "create", "like", "self", "various", "different", "needed", "essential",
    "requires", "clearly", "bring", "used", "fifth", "think", "require", "involved", "making", "said","main", "task", "continue",
    "work", "tenge", "necessary", "important", "aimed", "actively", "consider", "know", "single","responsible",
    "propose", "half", "include", "concrete", "greater", "carried", "show", "capable", "matter", "highly", "others", "solving",
    "turn", "currently", "word", "primarily", "type", "foremost", "related", "attention", "view", "time", "people",
    "sixth", "quite", "finally", "particularly", "president", "fact", "form"
]

In [7]:
df_test['full_text_clean'] = df_test['full_text_clean'].apply(
    lambda tokens: [word for word in tokens if word not in custom_stop and word not in other_stop]
)

In [8]:
df_test

,Title,Month,Year,Full Text,full_text_clean
0,President Kassym-Jomart Tokayev’s State of the...,September,2024,Dear Members of Parliament!\n\nCongratulations...,"[opening, regular, session, wish, success, act..."
1,President Kassym-Jomart Tokayev`s State of the...,September,2023,"Dear Compatriots, Distinguished Members of Par...","[distinguish, government, accordance, article,..."
2,President Kassym-Jomart Tokayev’s State of the...,September,2022,PRESIDENT KASSYM-JOMART TOKAYEV’S\n\nSTATE OF ...,"[state, nation, fair, state, nation, prosperou..."
3,State-of-the-Nation Address by President of th...,March,2022,NEW KAZAKHSTAN: THE PATH OF RENEWAL AND MODERN...,"[path, renewal, modernisation, friend, today, ..."
4,State of the Nation Address by President of th...,September,2021,UNITY OF THE PEOPLE AND SYSTEMIC REFORMS ARE A...,"[unity, systemic, reform, solid, foundation, n..."
5,President of Kazakhstan Kassym-Jomart Tokayev’...,September,2020,KAZAKHSTAN IN A NEW REALITY: TIME FOR ACTION\n...,"[reality, action, session, begin, challenge, c..."
6,President of Kazakhstan Kassym-Jomart Tokayev’...,September,2019,President Kassym-Jomart Tokayev’s State of the...,"[state, nation, constructive, public, dialogue..."
7,State of the Nation Address of President of th...,October,2018,GROWING WELFARE OF KAZAKH CITIZENS: INCREASE I...,"[grow, welfare, citizen, increase, income, qua..."
8,State of the Nation Address by the President o...,January,2018,NEW OPPORTUNITIES UNDER THE FOURTH INDUSTRIAL ...,"[opportunity, industrial, revolution, today, w..."
9,The President of Kazakhstan Nursultan Nazarbay...,January,2017,"""Third Modernization of Kazakhstan: Global Com...","[modernization, global, competitiveness, appro..."


In [9]:
df_test.sort_values(by=['Year', 'Month'], inplace=True)
df_test.reset_index(drop=True, inplace=True)

In [10]:
df_test

,Title,Month,Year,Full Text,full_text_clean
0,Address of the President of the Republic of Ka...,October,1997,"PROSPERITY, SECURITY\n\nAND EVER GROWING WELFA...","[prosperity, security, grow, welfare, message,..."
1,Address of the President of the Republic of Ka...,September,1998,On the Situation in the Country and major Dire...,"[situation, country, major, direction, domesti..."
2,Address of the President of the Republic of Ka...,September,1999,STABILITY AND SECURITY\nCOUNTRIES IN THE NEW C...,"[stability, security, country, century, genera..."
3,Address of the President of the Republic of Ka...,October,2000,"Towards Free, Effective and Secure Society\n\n...","[free, effective, secure, society, date, histo..."
4,Address of the President of the Republic of Ka...,September,2001,On the Situation in the Country and Main Direc...,"[situation, country, direction, domestic, fore..."
5,Address of the President of the Republic of Ka...,April,2002,ON THE MAIN DIRECTIONS OF DOMESTIC AND FOREIGN...,"[direction, domestic, foreign, policy, deputy,..."
6,Address of the President of the Republic of Ka...,April,2003,Our country continues to develop successfully ...,"[country, develop, successfully, accordance, s..."
7,Address of the President of the Republic of Ka...,March,2004,The past year has added to our confidence and ...,"[past, confidence, optimism, national, economi..."
8,Address of the President of the Republic of Ka...,February,2005,“Kazakhstan on the Road to Accelerated Economi...,"[road, accelerate, economic, social, political..."
9,Address of the President of the Republic of Ka...,March,2006,Kazakhstan’s strategy of joining the world’s 5...,"[strategy, join, world, competitive, country, ..."


In [11]:
all_tokens = [token for sublist in df_test["full_text_clean"] for token in sublist]
token_counts = Counter(all_tokens)
top_500 = token_counts.most_common(500)
top_500_df = pd.DataFrame(top_500, columns=["token", "count"])

In [12]:
top_500_df.to_csv("top_500_df_V2.csv", index=False)

In [13]:
df_test.columns

Index(['Title', 'Month', 'Year', 'Full Text', 'full_text_clean'], dtype='object')

In [14]:
sentences = list(df_test['full_text_clean'])
sentences

[['prosperity',
  'security',
  'grow',
  'welfare',
  'message',
  'country',
  'century',
  'virgil',
  'epoch',
  'goal',
  'heine',
  'introduction',
  'nothing',
  'remote',
  'yesterday',
  'nothing',
  'close',
  'tomorrow',
  'folk',
  'proverb',
  'fellow',
  'citizen',
  'appeal',
  'share',
  'vision',
  'future',
  'society',
  'mission',
  'state',
  'want',
  'present',
  'strategy',
  'sure',
  'help',
  'gain',
  'future',
  'accomplish',
  'mission',
  'wish',
  'share',
  'consideration',
  'future',
  'loom',
  'ahead',
  'century',
  'millenium',
  'pretty',
  'remote',
  'perspective',
  'future',
  'want',
  'build',
  'child',
  'believe',
  'ripen',
  'profound',
  'awareness',
  'longer',
  'live',
  'merely',
  'incessant',
  'turmoil',
  'settle',
  'present',
  'clear',
  'knowledge',
  'understand',
  'construct',
  'trajectory',
  'highway',
  'development',
  'ensure',
  'game',
  'objetives',
  'correct',
  'identification',
  'priority',
  'choose',
  '

In [15]:
from gensim.models.phrases import Phrases, Phraser

phrase_mod = Phrases(sentences, min_count=10, 
                  threshold=0.7, scoring = 'npmi')

print(phrase_mod[sentences[0]])

['prosperity', 'security', 'grow', 'welfare', 'message', 'country', 'century', 'virgil', 'epoch', 'goal', 'heine', 'introduction', 'nothing', 'remote', 'yesterday', 'nothing', 'close', 'tomorrow', 'folk', 'proverb', 'fellow', 'citizen', 'appeal', 'share', 'vision', 'future', 'society', 'mission', 'state', 'want', 'present', 'strategy', 'sure', 'help', 'gain', 'future', 'accomplish', 'mission', 'wish', 'share', 'consideration', 'future', 'loom', 'ahead', 'century', 'millenium', 'pretty', 'remote', 'perspective', 'future', 'want', 'build', 'child', 'believe', 'ripen', 'profound', 'awareness', 'longer', 'live', 'merely', 'incessant', 'turmoil', 'settle', 'present', 'clear', 'knowledge', 'understand', 'construct', 'trajectory', 'highway', 'development', 'ensure', 'game', 'objetives', 'correct', 'identification', 'priority', 'choose', 'relevant', 'strategy', 'manifest', 'firm', 'persistence', 'follow', 'path', 'dispense', 'unnecessary', 'track', 'waste', 'energy', 'resource', 'provide', 'st

In [16]:
count_dict = dict(phrase_mod.vocab)
count_dict

{'prosperity': 33,
 'security': 247,
 'prosperity_security': 4,
 'grow': 116,
 'security_grow': 2,
 'welfare': 28,
 'grow_welfare': 4,
 'message': 13,
 'welfare_message': 1,
 'country': 1125,
 'message_country': 1,
 'century': 102,
 'country_century': 4,
 'virgil': 1,
 'century_virgil': 1,
 'epoch': 9,
 'virgil_epoch': 1,
 'goal': 177,
 'epoch_goal': 1,
 'heine': 1,
 'goal_heine': 1,
 'introduction': 86,
 'heine_introduction': 1,
 'nothing': 28,
 'introduction_nothing': 1,
 'remote': 21,
 'nothing_remote': 1,
 'yesterday': 6,
 'remote_yesterday': 1,
 'yesterday_nothing': 1,
 'close': 37,
 'nothing_close': 1,
 'tomorrow': 10,
 'close_tomorrow': 1,
 'folk': 6,
 'tomorrow_folk': 1,
 'proverb': 7,
 'folk_proverb': 2,
 'fellow': 30,
 'proverb_fellow': 1,
 'citizen': 510,
 'fellow_citizen': 27,
 'appeal': 41,
 'citizen_appeal': 2,
 'share': 78,
 'appeal_share': 1,
 'vision': 39,
 'share_vision': 1,
 'future': 248,
 'vision_future': 8,
 'society': 364,
 'future_society': 4,
 'mission': 26,
 '

In [17]:
from gensim.models.phrases import npmi_scorer

npmi_scorer(count_dict['nurly'], count_dict['zhol'],
                count_dict['nurly_zhol'], len(phrase_mod.vocab),
                phrase_mod.min_count, phrase_mod.corpus_word_count)

0.955493225134465

In [18]:
bigram = dict() # пустой словарь для записи фильтрованных значений

for (key, value) in count_dict.items():
    if '_' in key:
        bigram[key] = value
        
bigram = pd.DataFrame.from_dict(bigram, orient = 'index').reset_index().rename(columns={"index": "Bigram", 0: "Count_B"})

In [19]:
top_500_bigrams = bigram.sort_values(by='Count_B', ascending=False).head(500)
top_500_bigrams

,Bigram,Count_B
13668,instruct_government,137
1413,economic_growth,101
7580,small_medium,67
14150,medium_size,65
7417,national_bank,59
...,...,...
2533,personnel_policy,8
15773,specific_proposal,8
1588,solution_problem,8
34286,akims_governor,8


In [20]:
scored_phrases = list(phrase_mod.export_phrases())
sorted_phrases = sorted(scored_phrases, key=lambda x: x[1], reverse=True)

sorted_phrases

['russia_china',
 'supreme_court',
 'european_union',
 'mutually_beneficial',
 'nurly_zhol',
 'otan_party',
 'prime_minister',
 'drug_traffic',
 'drinking_water',
 'transport_logistics',
 'private_partnership',
 'prosecutor_office',
 'artificial_intelligence',
 'square_metre',
 'solid_foundation',
 'socially_vulnerable',
 'corporate_governance',
 'inter_ethnic',
 'internal_affair',
 'small_medium',
 'electric_shock',
 'middle_east',
 'wide_range',
 'civil_servant',
 'oil_gas',
 'minimum_wage',
 'million_square',
 'shock_electric',
 'agro_industrial',
 'central_asian',
 'health_care',
 'territorial_integrity',
 'central_asia',
 'religious_extremism',
 'medium_size',
 'healthy_lifestyle',
 'large_scale',
 'labour_productivity',
 'samruk_kazyna']

In [21]:
proc_list = list(df_test['full_text_clean'])
proc_list = [phrase_mod[sublist] for sublist in proc_list]   
df_test['txt_clean_final'] = proc_list

In [22]:
df_test

,Title,Month,Year,Full Text,full_text_clean,txt_clean_final
0,Address of the President of the Republic of Ka...,October,1997,"PROSPERITY, SECURITY\n\nAND EVER GROWING WELFA...","[prosperity, security, grow, welfare, message,...","[prosperity, security, grow, welfare, message,..."
1,Address of the President of the Republic of Ka...,September,1998,On the Situation in the Country and major Dire...,"[situation, country, major, direction, domesti...","[situation, country, major, direction, domesti..."
2,Address of the President of the Republic of Ka...,September,1999,STABILITY AND SECURITY\nCOUNTRIES IN THE NEW C...,"[stability, security, country, century, genera...","[stability, security, country, century, genera..."
3,Address of the President of the Republic of Ka...,October,2000,"Towards Free, Effective and Secure Society\n\n...","[free, effective, secure, society, date, histo...","[free, effective, secure, society, date, histo..."
4,Address of the President of the Republic of Ka...,September,2001,On the Situation in the Country and Main Direc...,"[situation, country, direction, domestic, fore...","[situation, country, direction, domestic, fore..."
5,Address of the President of the Republic of Ka...,April,2002,ON THE MAIN DIRECTIONS OF DOMESTIC AND FOREIGN...,"[direction, domestic, foreign, policy, deputy,...","[direction, domestic, foreign, policy, deputy,..."
6,Address of the President of the Republic of Ka...,April,2003,Our country continues to develop successfully ...,"[country, develop, successfully, accordance, s...","[country, develop, successfully, accordance, s..."
7,Address of the President of the Republic of Ka...,March,2004,The past year has added to our confidence and ...,"[past, confidence, optimism, national, economi...","[past, confidence, optimism, national, economi..."
8,Address of the President of the Republic of Ka...,February,2005,“Kazakhstan on the Road to Accelerated Economi...,"[road, accelerate, economic, social, political...","[road, accelerate, economic, social, political..."
9,Address of the President of the Republic of Ka...,March,2006,Kazakhstan’s strategy of joining the world’s 5...,"[strategy, join, world, competitive, country, ...","[strategy, join, world, competitive, country, ..."


In [23]:
df_test.drop(columns=['full_text_clean'], inplace=True)

In [24]:
df_test

,Title,Month,Year,Full Text,txt_clean_final
0,Address of the President of the Republic of Ka...,October,1997,"PROSPERITY, SECURITY\n\nAND EVER GROWING WELFA...","[prosperity, security, grow, welfare, message,..."
1,Address of the President of the Republic of Ka...,September,1998,On the Situation in the Country and major Dire...,"[situation, country, major, direction, domesti..."
2,Address of the President of the Republic of Ka...,September,1999,STABILITY AND SECURITY\nCOUNTRIES IN THE NEW C...,"[stability, security, country, century, genera..."
3,Address of the President of the Republic of Ka...,October,2000,"Towards Free, Effective and Secure Society\n\n...","[free, effective, secure, society, date, histo..."
4,Address of the President of the Republic of Ka...,September,2001,On the Situation in the Country and Main Direc...,"[situation, country, direction, domestic, fore..."
5,Address of the President of the Republic of Ka...,April,2002,ON THE MAIN DIRECTIONS OF DOMESTIC AND FOREIGN...,"[direction, domestic, foreign, policy, deputy,..."
6,Address of the President of the Republic of Ka...,April,2003,Our country continues to develop successfully ...,"[country, develop, successfully, accordance, s..."
7,Address of the President of the Republic of Ka...,March,2004,The past year has added to our confidence and ...,"[past, confidence, optimism, national, economi..."
8,Address of the President of the Republic of Ka...,February,2005,“Kazakhstan on the Road to Accelerated Economi...,"[road, accelerate, economic, social, political..."
9,Address of the President of the Republic of Ka...,March,2006,Kazakhstan’s strategy of joining the world’s 5...,"[strategy, join, world, competitive, country, ..."


In [25]:
row1 = df_test.loc[15]
row2 = df_test.loc[16]

row2

Title              Address by the President of the Republic of Ka...
Month                                                       December
Year                                                            2012
Full Text          al and professional growth. The state and the ...
txt_clean_final    [growth, state, understand, instill, patriotis...
Name: 16, dtype: object

In [26]:
merged_row = row1.copy()
merged_row['Full Text'] = row1['Full Text'] + row2['Full Text']  
merged_row['txt_clean_final'] = row1['txt_clean_final'] + row2['txt_clean_final'] 
df_test = df_test.drop([15, 16])
df_test = pd.concat([
    df_test.iloc[:15],
    pd.DataFrame([merged_row]),
    df_test.iloc[15:]
]).reset_index(drop=True)

In [27]:
#df_test.to_csv("tm_no_splits.csv", index=False)

In [ ]:
#df_test['word_count_before'] = df_test['Full Text'].apply(lambda x: len(str(x).split()))
#df_test['word_count_after'] = df_test['txt_clean_final'].apply(len)

df_test

,Title,Month,Year,Full Text,txt_clean_final
0,Address of the President of the Republic of Ka...,October,1997,"PROSPERITY, SECURITY\n\nAND EVER GROWING WELFA...","[prosperity, security, grow, welfare, message,..."
1,Address of the President of the Republic of Ka...,September,1998,On the Situation in the Country and major Dire...,"[situation, country, major, direction, domesti..."
2,Address of the President of the Republic of Ka...,September,1999,STABILITY AND SECURITY\nCOUNTRIES IN THE NEW C...,"[stability, security, country, century, genera..."
3,Address of the President of the Republic of Ka...,October,2000,"Towards Free, Effective and Secure Society\n\n...","[free, effective, secure, society, date, histo..."
4,Address of the President of the Republic of Ka...,September,2001,On the Situation in the Country and Main Direc...,"[situation, country, direction, domestic, fore..."
5,Address of the President of the Republic of Ka...,April,2002,ON THE MAIN DIRECTIONS OF DOMESTIC AND FOREIGN...,"[direction, domestic, foreign, policy, deputy,..."
6,Address of the President of the Republic of Ka...,April,2003,Our country continues to develop successfully ...,"[country, develop, successfully, accordance, s..."
7,Address of the President of the Republic of Ka...,March,2004,The past year has added to our confidence and ...,"[past, confidence, optimism, national, economi..."
8,Address of the President of the Republic of Ka...,February,2005,“Kazakhstan on the Road to Accelerated Economi...,"[road, accelerate, economic, social, political..."
9,Address of the President of the Republic of Ka...,March,2006,Kazakhstan’s strategy of joining the world’s 5...,"[strategy, join, world, competitive, country, ..."


In [29]:
cov_df = pd.read_csv("covariets.csv")
df_merged = df_test.merge(cov_df, on=['Month', 'Year'], how='left')
cov_df

,Month,Year,GDP,Real GDP,FDI,inflation (in % to the previous month),Inlfation (% to the corresponding month of the previous year),unemployment rate (%)
0,October,1997,2.216593e+10,5.982470e+10,1.321400e+09,1.1,11.7,13.0
1,September,1998,2.213525e+10,5.868803e+10,1.151400e+09,-0.1,6.2,13.1
2,September,1999,1.687082e+10,6.027261e+10,1.587000e+09,0.7,12.8,13.5
3,October,2000,1.829199e+10,6.617933e+10,1.370521e+09,1.2,10.4,12.8
4,September,2001,2.215269e+10,7.511354e+10,2.816823e+09,0.2,8.0,10.5
5,April,2002,2.463660e+10,8.247466e+10,2.588491e+09,0.5,4.8,9.3
6,April,2003,3.083369e+10,9.014481e+10,2.483253e+09,0.4,7.0,8.8
7,March,2004,4.315165e+10,9.879871e+10,5.615263e+09,0.4,6.5,8.4
8,February,2005,5.712367e+10,1.083822e+11,2.546066e+09,0.6,6.9,8.1
9,March,2006,8.100386e+10,1.199791e+11,7.611168e+09,0.8,8.9,7.8


In [30]:
df_merged

,Title,Month,Year,Full Text,txt_clean_final,GDP,Real GDP,FDI,inflation (in % to the previous month),Inlfation (% to the corresponding month of the previous year),unemployment rate (%)
0,Address of the President of the Republic of Ka...,October,1997,"PROSPERITY, SECURITY\n\nAND EVER GROWING WELFA...","[prosperity, security, grow, welfare, message,...",2.216593e+10,5.982470e+10,1.321400e+09,1.1,11.7,13.0
1,Address of the President of the Republic of Ka...,September,1998,On the Situation in the Country and major Dire...,"[situation, country, major, direction, domesti...",2.213525e+10,5.868803e+10,1.151400e+09,-0.1,6.2,13.1
2,Address of the President of the Republic of Ka...,September,1999,STABILITY AND SECURITY\nCOUNTRIES IN THE NEW C...,"[stability, security, country, century, genera...",1.687082e+10,6.027261e+10,1.587000e+09,0.7,12.8,13.5
3,Address of the President of the Republic of Ka...,October,2000,"Towards Free, Effective and Secure Society\n\n...","[free, effective, secure, society, date, histo...",1.829199e+10,6.617933e+10,1.370521e+09,1.2,10.4,12.8
4,Address of the President of the Republic of Ka...,September,2001,On the Situation in the Country and Main Direc...,"[situation, country, direction, domestic, fore...",2.215269e+10,7.511354e+10,2.816823e+09,0.2,8.0,10.5
5,Address of the President of the Republic of Ka...,April,2002,ON THE MAIN DIRECTIONS OF DOMESTIC AND FOREIGN...,"[direction, domestic, foreign, policy, deputy,...",2.463660e+10,8.247466e+10,2.588491e+09,0.5,4.8,9.3
6,Address of the President of the Republic of Ka...,April,2003,Our country continues to develop successfully ...,"[country, develop, successfully, accordance, s...",3.083369e+10,9.014481e+10,2.483253e+09,0.4,7.0,8.8
7,Address of the President of the Republic of Ka...,March,2004,The past year has added to our confidence and ...,"[past, confidence, optimism, national, economi...",4.315165e+10,9.879871e+10,5.615263e+09,0.4,6.5,8.4
8,Address of the President of the Republic of Ka...,February,2005,“Kazakhstan on the Road to Accelerated Economi...,"[road, accelerate, economic, social, political...",5.712367e+10,1.083822e+11,2.546066e+09,0.6,6.9,8.1
9,Address of the President of the Republic of Ka...,March,2006,Kazakhstan’s strategy of joining the world’s 5...,"[strategy, join, world, competitive, country, ...",8.100386e+10,1.199791e+11,7.611168e+09,0.8,8.9,7.8


In [31]:
# Create boolean columns based on year
df_merged['is_Nazarbayev'] = (df_merged['Year'] <= 2018).astype(int)
df_merged['is_Tokayev'] = (df_merged['Year'] > 2018).astype(int)
df_merged

,Title,Month,Year,Full Text,txt_clean_final,GDP,Real GDP,FDI,inflation (in % to the previous month),Inlfation (% to the corresponding month of the previous year),unemployment rate (%),is_Nazarbayev,is_Tokayev
0,Address of the President of the Republic of Ka...,October,1997,"PROSPERITY, SECURITY\n\nAND EVER GROWING WELFA...","[prosperity, security, grow, welfare, message,...",2.216593e+10,5.982470e+10,1.321400e+09,1.1,11.7,13.0,1,0
1,Address of the President of the Republic of Ka...,September,1998,On the Situation in the Country and major Dire...,"[situation, country, major, direction, domesti...",2.213525e+10,5.868803e+10,1.151400e+09,-0.1,6.2,13.1,1,0
2,Address of the President of the Republic of Ka...,September,1999,STABILITY AND SECURITY\nCOUNTRIES IN THE NEW C...,"[stability, security, country, century, genera...",1.687082e+10,6.027261e+10,1.587000e+09,0.7,12.8,13.5,1,0
3,Address of the President of the Republic of Ka...,October,2000,"Towards Free, Effective and Secure Society\n\n...","[free, effective, secure, society, date, histo...",1.829199e+10,6.617933e+10,1.370521e+09,1.2,10.4,12.8,1,0
4,Address of the President of the Republic of Ka...,September,2001,On the Situation in the Country and Main Direc...,"[situation, country, direction, domestic, fore...",2.215269e+10,7.511354e+10,2.816823e+09,0.2,8.0,10.5,1,0
5,Address of the President of the Republic of Ka...,April,2002,ON THE MAIN DIRECTIONS OF DOMESTIC AND FOREIGN...,"[direction, domestic, foreign, policy, deputy,...",2.463660e+10,8.247466e+10,2.588491e+09,0.5,4.8,9.3,1,0
6,Address of the President of the Republic of Ka...,April,2003,Our country continues to develop successfully ...,"[country, develop, successfully, accordance, s...",3.083369e+10,9.014481e+10,2.483253e+09,0.4,7.0,8.8,1,0
7,Address of the President of the Republic of Ka...,March,2004,The past year has added to our confidence and ...,"[past, confidence, optimism, national, economi...",4.315165e+10,9.879871e+10,5.615263e+09,0.4,6.5,8.4,1,0
8,Address of the President of the Republic of Ka...,February,2005,“Kazakhstan on the Road to Accelerated Economi...,"[road, accelerate, economic, social, political...",5.712367e+10,1.083822e+11,2.546066e+09,0.6,6.9,8.1,1,0
9,Address of the President of the Republic of Ka...,March,2006,Kazakhstan’s strategy of joining the world’s 5...,"[strategy, join, world, competitive, country, ...",8.100386e+10,1.199791e+11,7.611168e+09,0.8,8.9,7.8,1,0


In [ ]:
#df_merged.to_csv("tm_cov_V2.csv", index=False)